### Ingestão — EPE-SEGOV/MS (publicações em PDF)

Arquitetura diferente das fontes de notícia: essa página não é um blog cronológico,
é uma galeria estática de documentos institucionais (manuais, políticas, relatórios).

- **Sem RSS, sem data de publicação visível na página.** A janela de tempo (24h)
  não se aplica — em vez disso, uso um **manifesto de URLs já processadas**,
  salvo à parte, pra só baixar/processar PDFs novos a cada execução.
- **Conteúdo é binário (PDF), não HTML de artigo.** Extração de texto usa `pypdf`,
  não `BeautifulSoup`.
- **Cadência esperada é baixa** (documentos mudam raramente) — rodar esse notebook
  diariamente é seguro (a dedupe evita reprocessar), mas não é daily news.


In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml pypdf
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# =============================================================================
# Imports
# =============================================================================
import os
import re
import io
import json
import time
import random
import hashlib
import unicodedata
import urllib.parse
from datetime import datetime, timezone
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests
from pypdf import PdfReader


In [0]:
# =============================================================================
# Configuração
# =============================================================================

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

SITE_URL = "https://www.epe.segov.ms.gov.br/publicacoes/"
SOURCE_ID = "epe_segov_ms"
SOURCE_DESCRICAO = "Linked from EPE-SEGOV/MS — Publicações"

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}/GERAL"
os.makedirs(PASTA_DESTINO, exist_ok=True)
print(f"[setup] Salvando artefatos em: {PASTA_DESTINO}")

# Manifesto de URLs já processadas — não é por data, é persistente entre execuções.
# É isso que substitui a "janela de 24h" que as fontes de notícia usam.
PASTA_MANIFESTOS = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/manifests"
os.makedirs(PASTA_MANIFESTOS, exist_ok=True)
CAMINHO_MANIFESTO = os.path.join(PASTA_MANIFESTOS, f"{SOURCE_ID}_processados.json")

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

HTTP_TIMEOUT = 30

# Mínimo de caracteres extraídos do PDF pra considerar válido (ex: PDF escaneado
# sem OCR pode vir vazio ou quase vazio).
MIN_CHARS_TEXTO = 200


[setup] Salvando artefatos em: /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-25


In [0]:
# =============================================================================
# Helpers
# =============================================================================

def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def carregar_manifesto(caminho: str) -> set:
    """URLs de PDF já processadas em execuções anteriores."""
    if not os.path.exists(caminho):
        return set()
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return set(json.load(f))
    except Exception as e:
        print(f"[manifesto] falha ao carregar ({e}); iniciando vazio.")
        return set()


def salvar_manifesto(caminho: str, urls: set) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(sorted(urls), f, ensure_ascii=False, indent=2)


In [0]:
# =============================================================================
# Etapa 1 — Baixar a página de listagem e extrair links de PDF
# =============================================================================

def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer="https://www.epe.segov.ms.gov.br/")

        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


def extrair_links_pdf(html: str, url_base: str) -> list[dict]:
    """
    Extrai só links que apontam pra arquivos .pdf — diferente das fontes de
    notícia, aqui não interessa link de menu/navegação, só documento.
    """
    soup = BeautifulSoup(html, "lxml")
    pdfs = []
    vistos = set()

    for tag_a in soup.find_all("a", href=True):
        href = tag_a["href"].strip()
        if not href.lower().endswith(".pdf"):
            continue

        url_absoluta = urllib.parse.urljoin(url_base, href)
        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)

        # Nome do arquivo na URL costuma ser mais informativo que o texto
        # da âncora (que aqui geralmente é só uma imagem, sem texto).
        nome_arquivo = urllib.parse.unquote(url_absoluta.split("/")[-1])
        titulo = re.sub(r"\.pdf$", "", nome_arquivo, flags=re.IGNORECASE)
        titulo = titulo.replace("-", " ").replace("_", " ").strip()

        pdfs.append({"titulo": titulo, "url": url_absoluta})

    return pdfs


In [0]:
# =============================================================================
# Etapa 2 — Baixar o PDF e extrair texto
# =============================================================================

def baixar_pdf(url: str, tentativas: int = 3) -> Optional[bytes]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            content_type = resp.headers.get("content-type", "")
            if resp.status_code == 200 and "pdf" in content_type.lower() and resp.content:
                return resp.content
            print(f"  [pdf tent {tentativa}/{tentativas}] status={resp.status_code} "
                  f"content-type={content_type!r}")
        except Exception as e:
            print(f"  [pdf tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.0))

    return None


def extrair_texto_pdf(conteudo_pdf: bytes) -> str:
    """
    Extrai texto de todas as páginas do PDF. Retorna string vazia se o PDF
    for escaneado sem OCR (nesse caso pypdf não consegue extrair nada) —
    tratado como falha na Etapa 3, não como erro fatal.
    """
    try:
        reader = PdfReader(io.BytesIO(conteudo_pdf))
        paginas = [p.extract_text() or "" for p in reader.pages]
        texto = "\n\n".join(paginas)
        return texto.strip()
    except Exception as e:
        print(f"    -> falha ao extrair texto do PDF: {e}")
        return ""


In [0]:
# =============================================================================
# Etapa 3 — Salvar no Volume
# =============================================================================

def salvar_artefatos(pasta: str, titulo: str, texto: str, metadados: dict) -> tuple[str, str]:
    slug_source = slugify(SOURCE_ID, max_len=40)
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")

    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json


In [0]:
# =============================================================================
# Etapa 4 — Pipeline principal
# =============================================================================

def processar_pdf(item: dict) -> Optional[dict]:
    titulo = item["titulo"]
    url = item["url"]
    print(f"\n  [pdf] {titulo[:100]}")

    conteudo = baixar_pdf(url)
    if not conteudo:
        print("    -> download do PDF falhou; pulando.")
        return None

    texto = extrair_texto_pdf(conteudo)
    if not texto or len(texto) < MIN_CHARS_TEXTO:
        print(f"    -> texto extraído insuficiente ({len(texto)} chars) — "
              f"possível PDF escaneado sem OCR; pulando.")
        return None

    # Schema canônico, com published_at ausente (a página não expõe data
    # de publicação por documento — limitação conhecida desta fonte).
    metadados = {
        "source_id": SOURCE_ID,
        "title": titulo,
        "description": SOURCE_DESCRICAO,
        "url": url,
        "date": HOJE,
        "published_at": None,
    }

    caminho_txt, caminho_json = salvar_artefatos(PASTA_DESTINO, titulo, texto, metadados)
    print(f"    -> salvo em {caminho_txt}")

    return {"titulo": titulo, "url": url, "caminho_txt": caminho_txt, "caminho_json": caminho_json}


In [0]:
# =============================================================================
# Execução
# =============================================================================

html = baixar_pagina(SITE_URL)
if not html:
    raise RuntimeError("Não foi possível baixar a página de listagem de publicações.")

pdfs_na_pagina = extrair_links_pdf(html, url_base=SITE_URL)
print(f"{len(pdfs_na_pagina)} PDFs encontrados na página.")

ja_processados = carregar_manifesto(CAMINHO_MANIFESTO)
pdfs_novos = [p for p in pdfs_na_pagina if p["url"] not in ja_processados]
print(f"{len(pdfs_novos)} PDFs novos (não presentes no manifesto de {len(ja_processados)} já processados).")

todos_resultados: list[dict] = []
for item in pdfs_novos:
    try:
        resultado = processar_pdf(item)
        if resultado:
            todos_resultados.append(resultado)
            ja_processados.add(item["url"])
    except Exception as e:
        print(f"[ERRO] PDF {item['titulo']!r} falhou: {e}")

salvar_manifesto(CAMINHO_MANIFESTO, ja_processados)

print(f"\n\n=== Fim. {len(todos_resultados)} PDFs novos processados e salvos em {PASTA_DESTINO} ===")
print(f"=== Manifesto atualizado: {len(ja_processados)} URLs conhecidas no total ===")


5 PDFs encontrados na página.
5 PDFs novos (não presentes no manifesto de 0 já processados).

  [pdf] Manual Value for Money 1
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-25/epe-segov-ms_manual-value-for-money-1_3c9fb659.txt

  [pdf] Manual Parcerias 1
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-25/epe-segov-ms_manual-parcerias-1_c01f6a23.txt

  [pdf] Relatorio de Auditoria
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-25/epe-segov-ms_relatorio-de-auditoria_5602ee32.txt

  [pdf] Declaracao de Apetite a Riscos EPE SEGOV 2026
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-25/epe-segov-ms_declaracao-de-apetite-a-riscos-epe-segov-2026_19a15026.txt

  [pdf] Politica de Gestao de Riscos EPE SEGOV 2026
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-25/epe-sego